In [3]:
import os
import shutil
from MRE import Job # Importamos el emulador

# Etiquetas para identificar de qué dataset viene cada equipo
TAG_PLATINIUM = "PLAT"
TAG_PREMIUM = "PREM"
TAG_ESTANDAR = "EST"

# --- FASE MAP ---
# Se ejecuta por cada línea. key = ID local, value = "ID_visitante ap_local ap_vis"
# Hay un map definido por dataset para etiquetar el origen (PLAT / PREM / EST).
# No se filtra por apuestas: solo importa en qué dataset aparece cada equipo.
# Cada dataset reclama equipos distintos: PLAT los locales, PREM los visitantes y EST ambos.

# Dataset Platinium: solo el equipo local
def fmapPlatinium(key, value, context):
    id_local = key  # ID_equipo_local es la clave
    context.write(id_local, TAG_PLATINIUM)

# Dataset Premium: solo el equipo visitante
def fmapPremium(key, value, context):
    datos = value.split() # separa la línea en campos
    id_visitante = datos[0]  # ID_equipo_visitante

    context.write(id_visitante, TAG_PREMIUM)

# Dataset Estandar: ambos equipos (local y visitante)
def fmapEstandar(key, value, context):
    datos = value.split() # separa la línea en campos
    id_local = key  # ID_equipo_local es la clave
    id_visitante = datos[0]  # ID_equipo_visitante

    context.write(id_local, TAG_ESTANDAR)
    context.write(id_visitante, TAG_ESTANDAR)

# --- FASE COMBINER ---
# Elimina etiquetas repetidas por equipo, solo con lo que vio su mapper.
# Reduce el tráfico de red. NO decide si el equipo está en un solo conjunto (ya que solo ve datos parciales).
def fcombiner(key, values, context):
    # set() deja una sola vez cada etiqueta; emitimos el mismo formato que el map
    for tag in set(values):
        context.write(key, tag)

# --- FASE REDUCE ---
# Recibe todas las etiquetas de UN equipo (key) ya agrupadas por el shuffle.
# Acá se decide si pertenece a exactamente uno de los tres conjuntos.
def fred(key, values, context):
    tags = set(values)

    plat = TAG_PLATINIUM in tags
    prem = TAG_PREMIUM in tags
    est = TAG_ESTANDAR in tags

   # Plat - (Prem U Est): está en Plat y NO está ni en Prem ni en Est
    solo_plat = plat and not (prem or est)

    # Prem - (Plat U Est): está en Prem y NO está ni en Plat ni en Est
    solo_prem = prem and not (plat or est)

    # Est - (Plat U Prem): está en Est y NO está ni en Plat ni en Prem
    solo_est = est and not (plat or prem)

    # Union de los tres conjuntos
    # Escribimos el equipo junto con el conjunto al que pertenece.
    if solo_plat:
        context.write(key, TAG_PLATINIUM)
    elif solo_prem:
        context.write(key, TAG_PREMIUM)
    elif solo_est:
        context.write(key, TAG_ESTANDAR)

# --- DRIVER ---
# Configura y dispara el job
if __name__ == '__main__':
    input_plat = "/content/in_platinium"   # carpeta con el dataset platinium
    input_prem = "/content/in_premium"     # carpeta con el dataset premium
    input_est = "/content/in_estandar"     # carpeta con el dataset estándar
    output_dir = "/content/salida_punto4"  # carpeta de resultados

    # Preparación: crear carpetas de entrada y copiar los archivos
    os.makedirs(input_plat, exist_ok=True)
    os.makedirs(input_prem, exist_ok=True)
    os.makedirs(input_est, exist_ok=True)
    shutil.copy("/content/apuestasPlatinium.txt", input_plat + "/apuestasPlatinium.txt")
    shutil.copy("/content/apuestasPremium.txt", input_prem + "/apuestasPremium.txt")
    shutil.copy("/content/apuestasEstandar.txt", input_est + "/apuestasEstandar.txt")

    # Borrar la salida anterior (si existe) para que no falle al re-ejecutar
    shutil.rmtree(output_dir, ignore_errors=True)

    job4 = Job(input_plat, output_dir, fmapPlatinium, fred) # Job con el primer dataset platinium y su map correspondiente
    job4.addInputPath(input_prem, fmapPremium) # Segundo dataset premium con su map
    job4.addInputPath(input_est, fmapEstandar) # Tercer dataset estándar con su map
    job4.setCombiner(fcombiner) # Combiner para reducir datos enviados por la red

    success = job4.waitForCompletion()

    if success:
        print("El job finalizó correctamente!")

El job finalizó correctamente!
